In [2]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import * 
spark = SparkSession.builder.appName('Telecom_CDR_Exercise').getOrCreate() 
# Read Dataset 
df = spark.read.csv('./telecom_cdr_production.csv/telecom_cdr_production.csv', header=True, inferSchema=True) 

In [4]:
df.printSchema()
df.show(5)
df.count()

root
 |-- cdr_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- roaming_type: string (nullable = true)
 |-- usage_amount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- sla_latency_ms: integer (nullable = true)
 |-- sla_breach_flag: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+-----------+------+------------+---------+-------------+------------+-------+--------------+---------------+--------------------+
| cdr_id|customer_id|region|service_type|plan_type| roaming_type|usage_amount|revenue|sla_latency_ms|sla_breach_flag|          event_time|
+-------+-----------+------+------------+---------+-------------+------------+-------+--------------+---------------+--------------------+
|3341057|      16717| North|        DATA|  PREPAID|INTERNATIONAL|      557.31|   8.36|           164|   

500000

Business Scenario 
Marketing needs top 5% revenue-generating customers. 
Problem Statement 
Segment customers into Platinum, Gold, Silver tiers. 
Business Requirements 
 Compute total revenue per customer 
 Rank customers 
 Label: 
 Platinum (Top 5%) 
 Gold (Next 15%) 
 Silver (Remaining) 
 Analyze roaming behavior of Platinum users

In [5]:
revenue = df.groupBy("customer_id").agg(sum("revenue").alias("total_revenue"))
revenue.show(5)

+-----------+-----------------+
|customer_id|    total_revenue|
+-----------+-----------------+
|      36131|89.46999999999998|
|      26623|88.58999999999999|
|      25517|           121.37|
|      17753|            23.86|
|      40574|           120.91|
+-----------+-----------------+
only showing top 5 rows



In [49]:
from pyspark.sql.window import Window
ranked = Window.orderBy(col("total_revenue").desc())

per_ranked = revenue.withColumn(
    "percent_rank", percent_rank().over(ranked)
)

per_ranked.show(5)

+-----------+------------------+------------+
|customer_id|     total_revenue|percent_rank|
+-----------+------------------+------------+
|      33699|385.78999999999996|         0.0|
|      34425|341.46000000000004|      2.5E-5|
|      23101|            328.35|      5.0E-5|
|      49180|324.04999999999995|      7.5E-5|
|      17125|            317.95|      1.0E-4|
+-----------+------------------+------------+
only showing top 5 rows



In [50]:
result = per_ranked.withColumn(
    "tier",
    when(col("percent_rank") <= 0.05, "Platinum")
    .when(col("percent_rank") <= 0.20, "Gold")
    .otherwise("Silver")
)

result.show(20)

+-----------+------------------+------------+--------+
|customer_id|     total_revenue|percent_rank|    tier|
+-----------+------------------+------------+--------+
|      33699|385.78999999999996|         0.0|Platinum|
|      34425|341.46000000000004|      2.5E-5|Platinum|
|      23101|            328.35|      5.0E-5|Platinum|
|      49180|324.04999999999995|      7.5E-5|Platinum|
|      17125|            317.95|      1.0E-4|Platinum|
|      30369|317.34999999999997|     1.25E-4|Platinum|
|      17558|            314.35|      1.5E-4|Platinum|
|      37128|313.27000000000004|     1.75E-4|Platinum|
|      26642|311.71000000000004|      2.0E-4|Platinum|
|      46805|             310.9|     2.25E-4|Platinum|
|      42432|310.53999999999996|      2.5E-4|Platinum|
|      30781|             308.5|     2.75E-4|Platinum|
|      46996|            305.98|      3.0E-4|Platinum|
|      22026|            304.88|     3.25E-4|Platinum|
|      26556|            304.55|      3.5E-4|Platinum|
|      158

In [51]:
analyze = result.groupBy("tier").agg(
    count("*").alias("customer_count"),
    round(sum("total_revenue"), 2).alias("tier_total_revenue"),
    round(avg("total_revenue"), 2).alias("avg_revenue")
)
analyze.show(20)

+--------+--------------+------------------+-----------+
|    tier|customer_count|tier_total_revenue|avg_revenue|
+--------+--------------+------------------+-----------+
|Platinum|          2001|         426287.29|     213.04|
|    Gold|          6000|         936699.67|     156.12|
|  Silver|         32000|        2591534.81|      80.99|
+--------+--------------+------------------+-----------+



In [52]:
platinum = result.filter(col("tier") == "Platinum").select("customer_id")
platinum.show(5)

+-----------+
|customer_id|
+-----------+
|      33699|
|      34425|
|      23101|
|      49180|
|      17125|
+-----------+
only showing top 5 rows



In [53]:
roaming_analyze = df.join(platinum, "customer_id").groupBy("roaming_type").agg(
    count("*").alias("event_count"),
    round(sum("revenue"), 2).alias("totale_revenue"),
    round(avg("revenue"), 2).alias("avg_revenue")
)
roaming_analyze.show(5)

+-------------+-----------+--------------+-----------+
| roaming_type|event_count|totale_revenue|avg_revenue|
+-------------+-----------+--------------+-----------+
|INTERNATIONAL|      14269|     314960.47|      22.07|
|         HOME|       9846|      55415.38|       5.63|
|     NATIONAL|       9949|      55911.44|       5.62|
+-------------+-----------+--------------+-----------+



2. Network Latency Trend Analysis 
Business Scenario 
Engineering wants service-level latency performance analysis. 
Problem Statement 
Determine which service suffers highest latency and SLA breach. 
Business Requirements 
 Compute average latency per service_type 
 Compute 95th percentile latency 
 Calculate breach ratio by service 
 Rank services by performance degradation 

In [54]:
avg_latency = df.groupBy("service_type").agg(
    round(avg("sla_latency_ms"), 2).alias("avg_latency_ms")
)
avg_latency.show(5)

+------------+--------------+
|service_type|avg_latency_ms|
+------------+--------------+
|         SMS|        274.83|
|        DATA|        275.05|
|       VOICE|        275.25|
+------------+--------------+



In [55]:
# agg = aggregate, withColumn is for calculating
breach_ratio = df.groupBy("service_type").agg(
    count("*").alias("total_events"),
    sum("sla_breach_flag").alias("breach_count")
).withColumn(
    "breach_ratio", round(col("breach_count") / col("total_events"), 2)
)
breach_ratio.show(5)

+------------+------------+------------+------------+
|service_type|total_events|breach_count|breach_ratio|
+------------+------------+------------+------------+
|         SMS|      166273|       73725|        0.44|
|        DATA|      166597|       74150|        0.45|
|       VOICE|      167130|       74331|        0.44|
+------------+------------+------------+------------+



In [56]:
result2 = df.groupBy("service_type").agg(
    round(percentile_approx("sla_latency_ms", 0.95), 2).alias("gt95_latency_ms")
)
result2.show(5)

+------------+---------------+
|service_type|gt95_latency_ms|
+------------+---------------+
|         SMS|            478|
|        DATA|            478|
|       VOICE|            478|
+------------+---------------+



In [57]:
latency_analysis = avg_latency.join(result2, "service_type").join(breach_ratio, "service_type")
latency_analysis.show(5)

+------------+--------------+---------------+------------+------------+------------+
|service_type|avg_latency_ms|gt95_latency_ms|total_events|breach_count|breach_ratio|
+------------+--------------+---------------+------------+------------+------------+
|         SMS|        274.83|            478|      166273|       73725|        0.44|
|        DATA|        275.05|            478|      166597|       74150|        0.45|
|       VOICE|        275.25|            478|      167130|       74331|        0.44|
+------------+--------------+---------------+------------+------------+------------+



In [60]:
window_spec = Window.orderBy(col("breach_ratio").desc(), col("gt95_latency_ms").desc())
latency_ranked = latency_analysis.withColumn(
    "degradation_rank", dense_rank().over(window_spec)
)

latency_ranked.orderBy("degradation_rank").show()


+------------+--------------+---------------+------------+------------+------------+----------------+
|service_type|avg_latency_ms|gt95_latency_ms|total_events|breach_count|breach_ratio|degradation_rank|
+------------+--------------+---------------+------------+------------+------------+----------------+
|        DATA|        275.05|            478|      166597|       74150|        0.45|               1|
|         SMS|        274.83|            478|      166273|       73725|        0.44|               2|
|       VOICE|        275.25|            478|      167130|       74331|        0.44|               2|
+------------+--------------+---------------+------------+------------+------------+----------------+



3. Roaming Revenue Mix Optimization 
Business Scenario 
International roaming is premium revenue source. 
Problem Statement 
Quantify revenue contribution by roaming_type. 
Business Requirements 
 Aggregate revenue by roaming_type 
 Compute revenue % share 
 Calculate average revenue per event 
 Identify highest unit revenue segment 

In [62]:
roaming_revenue = df.groupBy("roaming_type").agg(
    count("*").alias("event_count"),
    round(sum("revenue"), 2).alias("total_revenue"),
    round(avg("revenue"), 2).alias("avg_revenue_per_event")
)
roaming_revenue.show(5)

+-------------+-----------+-------------+---------------------+
| roaming_type|event_count|total_revenue|avg_revenue_per_event|
+-------------+-----------+-------------+---------------------+
|INTERNATIONAL|     166323|    2371262.2|                14.26|
|         HOME|     167169|    792519.32|                 4.74|
|     NATIONAL|     166508|    790740.25|                 4.75|
+-------------+-----------+-------------+---------------------+



In [74]:
total_rev = df.agg(sum("revenue")).collect()[0][0]

In [75]:
roaming_analysis = roaming_revenue.withColumn(
    "revenue_pct_share", round((col("total_revenue") / lit(total_rev)) * 100, 2)
)
roaming_analysis.show(5)

+-------------+-----------+-------------+---------------------+-----------------+
| roaming_type|event_count|total_revenue|avg_revenue_per_event|revenue_pct_share|
+-------------+-----------+-------------+---------------------+-----------------+
|INTERNATIONAL|     166323|    2371262.2|                14.26|            59.96|
|         HOME|     167169|    792519.32|                 4.74|            20.04|
|     NATIONAL|     166508|    790740.25|                 4.75|             20.0|
+-------------+-----------+-------------+---------------------+-----------------+



In [76]:
window_spec = Window.orderBy(col("avg_revenue_per_event").desc())
roaming_ranked = roaming_analysis.withColumn(
    "unit_revenue_rank", dense_rank().over(window_spec)
)

roaming_ranked.orderBy("unit_revenue_rank").show()

+-------------+-----------+-------------+---------------------+-----------------+-----------------+
| roaming_type|event_count|total_revenue|avg_revenue_per_event|revenue_pct_share|unit_revenue_rank|
+-------------+-----------+-------------+---------------------+-----------------+-----------------+
|INTERNATIONAL|     166323|    2371262.2|                14.26|            59.96|                1|
|     NATIONAL|     166508|    790740.25|                 4.75|             20.0|                2|
|         HOME|     167169|    792519.32|                 4.74|            20.04|                3|
+-------------+-----------+-------------+---------------------+-----------------+-----------------+

